# 12 - A3: frozen A2 + the JEV API

**Question this notebook answers:** given A2's frozen per-clip score, does asking the JEV API
typed questions over a small, label-free evidence state add anything (E5: A2-vs-A3), and what
would it cost to run this for real?

**A3 = frozen A2 + JEV.** A2 (a frozen-SSL detector) already produces one score per clip in
`experiments/<a2 exp>/scores/dev.csv`. A3 does not retrain anything: it builds a small JSON
"evidence state" per clip (A2's score/level plus a few acoustic summaries -- never the label,
attack id, speaker, path or clip id) and asks the JEV ("TypeSafe Jev API") three fixed,
versioned questions over that state:

1. **noul** -- "Does the evidence indicate synthetic speech?" (yes/no probabilities). This is
   the *only* question A3's own score file is derived from:
   `score = log(p(no) / p(yes))`, so higher = more bona fide, matching every other score file
   in this project.
2. **choice** -- bona fide / spoof / insufficient evidence.
3. **score** -- 0-3 evidence-consistency rating.

**Mode:** the JEV API's real request/response docs have not been shared yet, so
`TypesafeJevClient.ask` is an explicit `NotImplementedError` stub (see
`audio_df/jev/client.py`'s TODO list) and every JEV call in this notebook goes through
`DryRunJevClient` -- a deterministic, clearly-labelled stand-in (`route="dry-run"`,
`model_version="dry-run-0"`, `cost_usd=0.0`). **Nothing below is a real JEV result.** A2 has
not been trained yet either, so this notebook builds a small synthetic ASVspoof-shaped
fixture (`audio_df.synthetic.build_synthetic_asvspoof_tree`) and a fake A2 dev-scores file, so
every code path (state building, the cache, the runner) is exercised end-to-end today.

In [ ]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

from audio_df import data
from audio_df.synthetic import build_synthetic_asvspoof_tree


def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "pyproject.toml").exists():
            return cur
        cur = cur.parent
    raise RuntimeError(f"could not find repo root (pyproject.toml) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
print(f"REPO_ROOT = {REPO_ROOT}")

## Mode detection: real A2 dev scores, else a synthetic stand-in

Checks for a real, `done` A2 dev-scoring run at `configs/a3_jev.yaml: a2_experiment`. A2 is
not built yet (see `README.md` build status), so this always runs in **SYNTHETIC** mode
today; re-running once A2 lands switches automatically, no code changes.

In [ ]:
_a3_cfg_path = REPO_ROOT / "configs" / "a3_jev.yaml"
_a3_cfg = yaml.safe_load(_a3_cfg_path.read_text())
_a2_dir = REPO_ROOT / _a3_cfg["a2_experiment"]

USE_REAL_A2 = (_a2_dir / "scores" / "dev.csv").exists()

_tmpdir = tempfile.TemporaryDirectory()
_tmp_root = Path(_tmpdir.name)

if USE_REAL_A2:
    MODE = "REAL A2"
    a2_scores_path = _a2_dir / "scores" / "dev.csv"
    manifest_cfg = yaml.safe_load((REPO_ROOT / "configs" / "data.yaml").read_text())
    manifest = data.build_manifest(manifest_cfg)
else:
    MODE = "SYNTHETIC (no A2 run yet)"
    fixture_root = _tmp_root / "asvspoof_synth"
    synth_cfg = build_synthetic_asvspoof_tree(fixture_root, seed=1337, n_speakers_per_split=3)
    manifest = data.build_manifest(synth_cfg)
    dev_manifest = manifest[manifest["split"] == "dev"].reset_index(drop=True)

    # A fake A2: bonafide clips score positive, spoof negative, with a little noise -- just
    # enough for this notebook's dry-run demo to exercise every code path.
    rng = np.random.default_rng(0)
    fake_scores = pd.DataFrame(
        {
            "clip_id": dev_manifest["utt_id"],
            "label": dev_manifest["label"],
            "score": [
                (0.6 if lbl == "bonafide" else -0.6) + rng.normal(0, 0.1)
                for lbl in dev_manifest["label"]
            ],
            "attack_id": dev_manifest["attack_id"],
        }
    )
    a2_exp_dir = _tmp_root / "fake_a2_exp"
    (a2_exp_dir / "scores").mkdir(parents=True)
    fake_scores.to_csv(a2_exp_dir / "scores" / "dev.csv", index=False)
    a2_scores_path = a2_exp_dir / "scores" / "dev.csv"

    dev_manifest_path = _tmp_root / "dev_manifest.csv"
    dev_manifest.to_csv(dev_manifest_path, index=False)

print(f"MODE = {MODE}")
print(f"a2 dev scores: {a2_scores_path}")

`configs/a3_jev.yaml`'s `thresholds.detector.eer_threshold` defaults to `null` --
    read from `<a2_experiment>/metrics.json` by `run_a3` (`audio_df.experiments.run_a3.
    resolve_eer_threshold`), which does not exist for this notebook's synthetic fake-A2
    fixture. This demo therefore uses an explicit override (`0.0`, matching the fake
    scores' bonafide/spoof sign convention above) purely so `build_state` has a threshold
    to compare against -- **not** a real A2 dev EER threshold.

In [ ]:
demo_cfg = dict(_a3_cfg)
demo_cfg["thresholds"] = dict(_a3_cfg["thresholds"])
demo_cfg["thresholds"]["detector"] = {**_a3_cfg["thresholds"]["detector"], "eer_threshold": 0.0}
print(f"demo eer_threshold override = {demo_cfg['thresholds']['detector']['eer_threshold']}")

## One example evidence state, both variants

`audio_df.jev.state.build_state` turns one clip's A2 score + audio into a small,
JSON-serialisable, **label-free** dict. `"numbers"` keeps raw floats; `"words"` buckets
everything into threshold words (thresholds from `configs/a3_jev.yaml: thresholds`, applied
in code, never learned).

In [ ]:
from audio_df.audio import load_audio
from audio_df.jev.state import build_state

a2_scores = pd.read_csv(a2_scores_path)
example_row = a2_scores.iloc[0]
example_clip_id = example_row["clip_id"]
example_audio_path = dev_manifest.loc[
    dev_manifest["utt_id"] == example_clip_id, "path"
].iloc[0]
example_audio, _sr = load_audio(example_audio_path, target_sr=demo_cfg["sample_rate"])

state_numbers = build_state(example_clip_id, float(example_row["score"]), example_audio, demo_cfg, variant="numbers")
state_words = build_state(example_clip_id, float(example_row["score"]), example_audio, demo_cfg, variant="words")

print("numbers variant:")
print(json.dumps(state_numbers, indent=2))
print("\nwords variant:")
print(json.dumps(state_words, indent=2))

Note what is *not* in either dict: `example_clip_id`, the file path, the label, and the
attack id never appear -- see `tests/test_jev_state.py::test_no_label_leakage`, which asserts
this programmatically.

## The three questions

Versioned text (`audio_df.jev.questions.QUESTION_TEXT_VERSION`), so a wording change
invalidates cached answers rather than silently mixing them in with answers to the old
wording.

In [ ]:
from audio_df.jev.questions import CHOICE_QUESTION, NOUL_QUESTION, QUESTION_SETS, SCORE_QUESTION

for q in (NOUL_QUESTION, CHOICE_QUESTION, SCORE_QUESTION):
    print(f"[{q.id:>6}] ({q.question_type}, {q.version}) {q.text}")
    if q.options:
        print(f"          options: {q.options}")
    if q.scale:
        print(f"          scale:   {q.scale}")

print("\nquestion sets:", {k: [q.id for q in v] for k, v in QUESTION_SETS.items()})

## A dry run on 5 clips

Every call below goes through `DryRunJevClient` -- deterministic, `route="dry-run"`,
`cost_usd=0.0`. This exercises the full runner (`audio_df.experiments.run_a3.run_a3`): the
cache, the state builder, both score files, `answers.csv`, and the cost/latency summary.

In [ ]:
from audio_df.experiments.run_a3 import run_a3

demo_cfg["a2_experiment"] = str(a2_scores_path.parent.parent)
demo_cfg["subset"] = {
    "manifest": str(dev_manifest_path),
    "n": 5,
    "seed": 1337,
    "strata": ["label", "attack_id"],
}
demo_cfg["cache_dir"] = str(_tmp_root / "jev_cache")

exp_dir = run_a3(
    demo_cfg,
    state_variant="numbers",
    question_set="noul_only",
    client_kind="dry-run",
    root=str(_tmp_root / "experiments"),
    lock_path=str(_tmp_root / "lock"),
    status_fn=lambda: {
        "pressure_level": 1,
        "pressure_name": "normal",
        "swap_used_gb": 0.1,
        "swap_total_gb": 1.0,
        "rss_mb": 1.0,
    },
)
print(f"exp_dir = {exp_dir}")
pd.read_csv(exp_dir / "scores" / "jev_dev500.csv")

In [ ]:
summary = json.loads((exp_dir / "cost_summary.json").read_text())
print(json.dumps(summary, indent=2, sort_keys=True))

### Observation

Every row above got a score; `cost_summary.json` reports 0 real dollars spent
(`DryRunJevClient` is free by construction) and every latency is a few microseconds of
in-process computation -- **none of this is evidence about the real JEV API's latency or
cost.**

### Inference

The plumbing (state -> cache -> client -> score file) works end-to-end on synthetic data.
That is the only claim this cell supports.

## The plan once `TypesafeJevClient.ask` is implemented

1. **Measure first, on <= 20 real calls.** `run_a3`'s budget guard already refuses
   `--client typesafe` runs above `n=20` until a real cost-per-call is known (see
   `audio_df/experiments/run_a3.py`'s `_TYPESAFE_FIRST_RUN_MAX_N`). Run all four
   `(state, question_set)` combinations once each at `n<=20`, read the printed
   `cost_summary.json` (`cost_mean_usd`, `latency_median_s`, `latency_p90_s`), and only then
   decide the real subset size (the config default is 500, per E5's plan) against a real
   budget.
2. **Then scale up** with `--n 500` (or whatever the measured budget affords) and `--resume`
   if a run is interrupted -- the cache means resuming never re-pays for an already-answered
   question.
3. **Compare A2 vs A3** on the identical clip set (`scores/a2_dev500.csv` vs
   `scores/jev_dev500.csv`, written by the same run) via
   `scripts/score_experiment.py` + `scripts/make_tables.py`'s E5 table.

### The four variant commands

(`state` x `question_set`; run each once at `n<=20` first per the plan above)

In [ ]:
for state in _a3_cfg["variants"]["state"]:
    for question_set in _a3_cfg["variants"]["question_set"]:
        print(
            "uv run python -m audio_df.experiments.run_a3 run "
            f"--state {state} --questions {question_set} --client dry-run --n 20"
        )

## After a real run: A2 vs A3, and answer distributions

Not executable yet (no real A2 or JEV run exists) -- this is the analysis this notebook will
run once both exist, kept here as the concrete next step rather than a placeholder:

```python
a2 = pd.read_csv(exp_dir / "scores" / "a2_dev500.csv")
a3 = pd.read_csv(exp_dir / "scores" / "jev_dev500.csv")
merged = a2.merge(a3, on=["clip_id", "label"], suffixes=("_a2", "_a3"))
merged[["score_a2", "score_a3"]].corr()

answers = pd.read_csv(exp_dir / "answers.csv")
answers.groupby("question_id")["result"].apply(lambda s: s.value_counts().head())
```

Then: `uv run python scripts/score_experiment.py <exp_dir>` and
`uv run python scripts/make_tables.py` to regenerate the E5 (A2-vs-A3) table.

In [ ]:
_tmpdir.cleanup()